# Exercício 04 — FewShot Marketing

**Versão didática:** neste caderno está **todo** o fluxo (Pydantic, prompt few-shot LangChain, Instructor + Gemini) — **sem** `from app…`. A pasta [`app/`](app/) espelha o mesmo para Docker/FastAPI.

Documentação: [`README.md`](README.md), [`docs/explicacao_teorica.md`](docs/explicacao_teorica.md).

## 0. Ambiente

`GOOGLE_API_KEY` no `.env` na raiz do repositório do curso.

In [ ]:
from pathlib import Path
import os
from dotenv import load_dotenv

EX_ROOT = Path.cwd().resolve()
REPO_ROOT = EX_ROOT.parent.parent

load_dotenv(REPO_ROOT / ".env", override=False)
load_dotenv(EX_ROOT / ".env", override=True)

if not (os.environ.get("GOOGLE_API_KEY") or os.environ.get("GEMINI_API_KEY")):
    raise RuntimeError("Defina GOOGLE_API_KEY no .env na raiz do repositório.")

print("OK — exercício:", EX_ROOT)

## 1. O que é *few-shot* aqui?

Incluímos **três campanhas exemplo** no *system* para ancorar ritmo e estrutura. O modelo gera uma campanha **nova** para o teu briefing, mas «no mesmo espírito». **Instructor** obriga a resposta a validar o schema Pydantic `Campanha`.

## 2. Schema Pydantic + presets de estilo (`estilo`)

In [ ]:
from typing import Literal

from pydantic import BaseModel, Field, field_validator, model_validator

EstiloPreset = Literal["livre", "formal", "engracado", "tecnico", "provocativo"]

ESTILO_PARA_INSTRUCAO: dict[str, str] = {
    "livre": "Segue apenas as notas de tom indicadas pelo cliente (sem preset rígido).",
    "formal": "Tom institucional e credível; vocabulário cuidado; evita gírias e ironia pesada.",
    "engracado": "Tom leve e espirituoso (nível LinkedIn/Twitter profissional); sem crueldade nem estereótipos.",
    "tecnico": "Tom preciso com termos da área; assume público que tolera detalhe; poucos adjetivos vazios.",
    "provocativo": "Contraste forte, perguntas incómodas e urgência inteligente — sem insultos, discriminação ou desinformação.",
}

class Campanha(BaseModel):
    titulo: str = Field(..., max_length=140)
    publico_alvo: str
    tom: str
    texto_post: str = Field(..., min_length=40)
    chamada_para_acao: str = Field(..., min_length=4)
    hashtags: list[str] = Field(..., min_length=3, max_length=12)

    @field_validator("hashtags", mode="before")
    @classmethod
    def normalizar_hashtags(cls, v: object) -> object:
        if not isinstance(v, list):
            return v
        out = []
        for item in v:
            if not isinstance(item, str):
                continue
            t = item.strip()
            if not t:
                continue
            if not t.startswith("#"):
                t = "#" + t.lstrip("#")
            out.append(t)
        return out

    @model_validator(mode="after")
    def texto_ok(self) -> "Campanha":
        if len(self.texto_post.strip()) < 40:
            raise ValueError("texto_post demasiado curto")
        return self

class GerarCampanhaEntrada(BaseModel):
    produto: str = Field(..., min_length=3)
    publico: str = Field(..., min_length=3)
    tom: str = Field(default="didático e próximo")
    estilo: EstiloPreset = Field(default="livre")

print("Presets:", list(ESTILO_PARA_INSTRUCAO.keys()))

## 3. LangChain (`ChatPromptTemplate`) + Instructor (`from_provider`)

- Montamos mensagens **system** + **human**.
- Passamos para `instructor.from_provider(f"google/{modelo}").create(..., response_model=Campanha)`.

In [ ]:
from functools import lru_cache

import instructor
from langchain_core.messages import BaseMessage
from langchain_core.prompts import ChatPromptTemplate

_EXEMPLOS_FEWSHOT = """
### Exemplo de campanha A (marca directa, tom profissional-quente)
- titulo: «O teu CRM não precisa de mais dashboards — precisa de menos drama»
- publico_alvo: líderes de vendas B2B em scale-ups europeias
- tom: confiante e irónico leve
- texto_post: «Durante anos vendemos a promessa do «360º vision». O resultado? Equipas a passar horas a actualizar campos que ninguém lê.
- chamada_para_acao: «Marca uma demo de 15 min e leva o checklist anti-reunião vazio.»
- hashtags: ["#SalesOps", "#B2B", "#Produtividade"]

### Exemplo B (didático, público técnico)
- titulo: «Embeddings não são magia — são geometria com pressa de entrega»
- publico_alvo: engenheiros de software a entrar em IA aplicada
- tom: didático, metáforas técnicas
- texto_post: «Se pensas em embeddings como números misteriosos…»
- chamada_para_acao: «Descarrega o notebook-base de chunking.»
- hashtags: ["#RAG", "#Embeddings", "#ML"]

### Exemplo C (provocativo-controlado)
- titulo: «Compliance não é PowerPoint — é decisão sob pressão»
- publico_alvo: DPOs e security leads em SaaS
- tom: provocativo mas respeitoso
- texto_post: «Ninguém compra software de compliance por paixão…»
- chamada_para_acao: «Pede o template de runbook de incidente.»
- hashtags: ["#Compliance", "#Security", "#SaaS"]
"""

_SYSTEM_BASE = f"""És copywriter sénior da **FewShot Marketing** (Portugal — português europeu).
Inspiras-te nos exemplos; saída validada por schema Pydantic.
`hashtags`: 3–12 com `#`. Sem discriminação nem claims ilegais.

{_EXEMPLOS_FEWSHOT}"""

_PROMPT = ChatPromptTemplate.from_messages([
    ("system", _SYSTEM_BASE + "\n### Instruções de estilo neste pedido\n{instrucoes_estilo}\n"),
    ("human",
     "Novo briefing:\n"
     "- Produto/serviço: {produto}\n"
     "- Público-alvo: {publico}\n"
     "- Notas adicionais de tom: {tom}\n\n"
     "Gera **uma** campanha nova no mesmo espírito dos exemplos."),
])

def _messages_para_dicts(messages: list[BaseMessage]) -> list[dict[str, str]]:
    role_map = {"system": "system", "human": "user", "ai": "assistant"}
    return [{"role": role_map.get(getattr(m, "type", ""), "user"), "content": str(m.content)} for m in messages]

def _nome_modelo() -> str:
    raw = (os.environ.get("GEMINI_MODEL_EX04") or os.environ.get("GEMINI_MODEL") or "gemini-2.0-flash").replace("models/", "")
    return raw.strip()

@lru_cache(maxsize=1)
def _cliente_instructor():
    return instructor.from_provider(f"google/{_nome_modelo()}")

def montar_instrucoes_estilo(body: GerarCampanhaEntrada) -> str:
    return ESTILO_PARA_INSTRUCAO.get(body.estilo, ESTILO_PARA_INSTRUCAO["livre"])

def gerar_campanha(body: GerarCampanhaEntrada) -> Campanha:
    msgs = _PROMPT.format_messages(
        produto=body.produto.strip(),
        publico=body.publico.strip(),
        tom=body.tom.strip(),
        instrucoes_estilo=montar_instrucoes_estilo(body),
    )
    payload = _messages_para_dicts(msgs)
    return _cliente_instructor().create(messages=payload, response_model=Campanha, max_retries=2)

print("Prompt + cliente Instructor definidos.")

## 4. Exemplo do enunciado

In [ ]:
import json

entrada = GerarCampanhaEntrada(
    produto="curso de agentes inteligentes",
    tom="didático e sarcástico",
    publico="profissionais de tecnologia",
    estilo="livre",
)
print(json.dumps(entrada.model_dump(), ensure_ascii=False, indent=2))

camp = gerar_campanha(entrada)
print("\n", camp.model_dump_json(indent=2, ensure_ascii=False))

## 5. Desafio extra — preset `provocativo`

In [ ]:
alt = GerarCampanhaEntrada(
    produto="curso de agentes inteligentes",
    publico="profissionais de tecnologia",
    tom="urgência inteligente, sem clickbait baixo",
    estilo="provocativo",
)
camp2 = gerar_campanha(alt)
print(camp2.titulo)
print("---")
print(camp2.texto_post[:700], "…")

## Referências

- [`app/main.py`](app/main.py) — API `POST /campanhas/gerar`
- [`app/schemas/campanha.py`](app/schemas/campanha.py) — espelho do schema
- [`app/services/gerador_campanha.py`](app/services/gerador_campanha.py) — espelho da lógica